<a href="https://colab.research.google.com/github/VincentMunizAdrian/Ejercicio-Data-Science-III---stopwords-con-spaCy/blob/main/Ejercicio_Stopwords_con_spaCy_Unidad_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introducción

**Objetivo**

El objetivo de esta práctica es aplicar técnicas de procesamiento de lenguaje natural (NLP) utilizando spaCy para realizar la tokenización de textos y la eliminación de stopwords. Además, se busca evaluar cómo la personalización de stopwords puede mejorar la representación textual de un conjunto de reseñas de productos de Amazon.

## Descripción del dataset

Se utilizó el dataset Amazon Reviews Dataset disponible en Kaggle. Este conjunto de datos contiene reseñas de usuarios sobre productos comercializados en Amazon, incluyendo texto libre escrito por los compradores.

Para el análisis se trabajó con la columna que contiene el texto de las reseñas, ya que representa la fuente principal de información para las tareas de procesamiento de lenguaje natural.

## Instalación e importación

In [31]:
# 1. Instalación de librerías necesarias

# spaCy y el modelo en español no siempre vienen preinstalados en Colab
!pip install spacy
!python -m spacy download es_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 48.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 67.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [32]:
# 2. Importación de librerías

import pandas as pd
import numpy as np
import spacy
from collections import Counter
from google.colab import files

## Carga del modelo

In [33]:
# 3. Carga del modelo de spaCy (español)

nlp = spacy.load("en_core_web_sm")

## Subida y carga del dataset

In [34]:
# 4. Subida del dataset desde la PC

# El dataset debe descargarse previamente desde Kaggle
# https://www.kaggle.com/datasets/dongrelaxman/amazon-reviews-dataset
# y subirse manualmente como archivo CSV

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving Amazon_Reviews.csv to Amazon_Reviews (1).csv


In [35]:
# 5. Carga robusta del CSV (soluciona ParserError)
df = pd.read_csv(
    file_name,
    sep=",",
    quotechar='"',
    encoding="utf-8",
    engine="python"
)

In [36]:
# 6. Inspección inicial del dataset

print("Dimensiones del dataset:")
print(df.shape)

print("\nColumnas disponibles:")
print(df.columns)

print("\nPrimeras filas del dataset:")
df.head()

Dimensiones del dataset:
(21214, 9)

Columnas disponibles:
Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

Primeras filas del dataset:


,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


## Seleccionar la columna de texto

### Normalización del texto

In [37]:
import re

texts = df["Review Text"].dropna().astype(str)

# Muestra
texts = texts.sample(1000, random_state=42)

def normalize_text(text):
    text = text.lower()                    # minúsculas
    text = re.sub(r'\s+', ' ', text)       # espacios múltiples
    return text.strip()

texts = texts.apply(normalize_text)

## Tokenización básica con spaCy

In [38]:
sample_text = texts.iloc[0]

doc = nlp(sample_text)

tokens = [token.text for token in doc]

print(tokens[:50])

['disgusting', '!', 'i', 'ordered', 'an', 'expensive', 'item', 'with', 'amazon', 'and', 'made', 'sure', 'it', 'was', 'a', '‘', 'prime', '’', 'item', 'so', 'that', 'i', 'had', 'the', 'next', 'day', 'deliver', 'promise', '...', 'not', 'only', 'did', 'i', 'receive', 'the', 'item', '3days', 'later', '(', 'not', 'next', 'day', '!', ')', 'i', 'was', 'also', 'charged', 'twice', '!']


**Qué se hizo**

Se aplicó el modelo en_core_web_sm de spaCy para segmentar cada reseña en unidades léxicas (tokens). La tokenización constituye una etapa fundamental del procesamiento de texto, ya que permite analizar cada palabra individualmente.

**Ejemplo**

*Texto original:*

This product is really good and easy to use.

*Tokens obtenidos:*

['This', 'product', 'is', 'really', 'good', 'and', 'easy', 'to', 'use', '.']

## Eliminación de stopwords estándar

In [39]:
def clean_standard(text):
    doc = nlp(text)

    return [
        token.text
        for token in doc
        if token.is_alpha
        and not token.is_stop
    ]

In [40]:
# Verificamos
ejemplo = texts.iloc[0]

print("Texto original:")
print(ejemplo)

print("\nTokens limpios:")
print(clean_standard(ejemplo)[:50])

Texto original:
disgusting! i ordered an expensive item with amazon and made sure it was a ‘prime’ item so that i had the next day deliver promise... not only did i receive the item 3days later (not next day!) i was also charged twice! upon speaking to their customer service team, i was told they had made a mistake and sent the parcel in the opposite direction from my address! (which happens) as for being charged twice, i apparently ordered two and cancelled one! which was definitely not the case!!!!! when i asked the person i spoke to about paying for prime and my next day delivery, they had no response whatsoever! in fact they were actually very rude!! so why am i paying for it?! i have always noticed that my items usually aren’t next day, but i never think to much about it as they are usually just little cheap things for my children, however this item was expensive and was needed literally the next day!! poor poor service from amazon recently! have really questioned using them latel

**Qué se hizo**

Se utilizaron las stopwords predefinidas por spaCy para el idioma inglés. Estas palabras son extremadamente frecuentes en los textos y suelen aportar poca información semántica para tareas analíticas.

**Justificación**

Palabras como "the", "and", "is", "to" o "of" aparecen en gran cantidad de documentos, pero generalmente no ayudan a distinguir el contenido específico de una reseña.

**Ejemplo**

*Antes:*

This product is really good and easy to use

*Después:*

product really good easy use

## Comparación de frecuencias

### Antes de eliminar stopwords

Las palabras más frecuentes corresponden principalmente a términos funcionales del idioma inglés.

In [41]:
all_tokens_before = []

for text in texts:
    doc = nlp(text)

    all_tokens_before.extend(
        [
            token.text.lower()
            for token in doc
            if token.is_alpha
        ]
    )

freq_before = Counter(all_tokens_before)

print("20 palabras más frecuentes:")
print(freq_before.most_common(20))

20 palabras más frecuentes:
[('i', 3138), ('the', 2912), ('to', 2745), ('and', 2367), ('a', 1807), ('amazon', 1465), ('it', 1295), ('they', 1256), ('my', 1162), ('of', 1074), ('for', 1000), ('is', 994), ('that', 903), ('not', 824), ('was', 810), ('on', 807), ('have', 805), ('you', 759), ('in', 740), ('me', 683)]


### Después de eliminar stopwords estándar

Comienzan a aparecer términos asociados a características de los productos y opiniones de los usuarios.

In [42]:
all_tokens_after = []

for text in texts:
    all_tokens_after.extend(clean_standard(text))

freq_after = Counter(all_tokens_after)

print("20 palabras más frecuentes:")
print(freq_after.most_common(20))

20 palabras más frecuentes:
[('amazon', 1465), ('customer', 529), ('service', 513), ('delivery', 400), ('order', 351), ('time', 293), ('item', 283), ('prime', 276), ('account', 272), ('day', 240), ('money', 223), ('refund', 215), ('items', 204), ('delivered', 185), ('company', 184), ('ordered', 172), ('days', 167), ('told', 165), ('card', 151), ('like', 146)]


## Personalización de stopwords

### Despues de stopwords personalizadas

La eliminación de términos específicos del dominio permite que palabras más representativas del contenido ocupen las primeras posiciones del ranking.

In [45]:
custom_stopwords = {
    "product",
    "amazon",
    "buy",
    "bought",
    "service"
}

for word in custom_stopwords:
    nlp.vocab[word].is_stop = True

In [46]:
def clean_custom(text):
    doc = nlp(text)

    return [
        token.text
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and token.text not in custom_stopwords
    ]

In [47]:
all_tokens_custom = []

for text in texts:
    all_tokens_custom.extend(clean_custom(text))

freq_custom = Counter(all_tokens_custom)

print("20 palabras más frecuentes:")
print(freq_custom.most_common(20))

20 palabras más frecuentes:
[('customer', 529), ('delivery', 400), ('order', 351), ('time', 293), ('item', 283), ('prime', 276), ('account', 272), ('day', 240), ('money', 223), ('refund', 215), ('items', 204), ('delivered', 185), ('company', 184), ('ordered', 172), ('days', 167), ('told', 165), ('card', 151), ('like', 146), ('got', 141), ('use', 138)]


Durante el análisis exploratorio se observó que estas palabras aparecían con mucha frecuencia en las reseñas debido a la naturaleza del dataset. Sin embargo, su presencia no aporta información relevante sobre la opinión del usuario respecto al producto.

Por ejemplo, la palabra "product" aparece en una gran cantidad de comentarios independientemente de que la valoración sea positiva o negativa. Del mismo modo, términos como "amazon", "buy" y "bought" están relacionados con el contexto de compra más que con la experiencia o satisfacción del cliente.

Por este motivo se decidió incorporarlas a la lista de stopwords personalizadas para reducir ruido y mejorar la calidad de la representación textual.

## Comparación cuantitativa

In [48]:
print("Cantidad de tokens originales:",
      len(all_tokens_before))

print("Cantidad luego de limpieza estándar:",
      len(all_tokens_after))

print("Cantidad luego de limpieza personalizada:",
      len(all_tokens_custom))

Cantidad de tokens originales: 82199
Cantidad luego de limpieza estándar: 34554
Cantidad luego de limpieza personalizada: 32270


## Visualización

In [49]:
import pandas as pd

top_before = pd.DataFrame(
    freq_before.most_common(10),
    columns=["Palabra", "Frecuencia"]
)

top_after = pd.DataFrame(
    freq_after.most_common(10),
    columns=["Palabra", "Frecuencia"]
)

print("ANTES")
display(top_before)

print("DESPUÉS")
display(top_after)

ANTES


,Palabra,Frecuencia
0,i,3138
1,the,2912
2,to,2745
3,and,2367
4,a,1807
5,amazon,1465
6,it,1295
7,they,1256
8,my,1162
9,of,1074


DESPUÉS


,Palabra,Frecuencia
0,amazon,1465
1,customer,529
2,service,513
3,delivery,400
4,order,351
5,time,293
6,item,283
7,prime,276
8,account,272
9,day,240


## Conclusión

La eliminación de stopwords permitió reducir significativamente la cantidad de términos presentes en el corpus, eliminando palabras funcionales como artículos, pronombres y preposiciones que aportan poco significado semántico. Al analizar las frecuencias antes de la limpieza, predominaban términos muy comunes como "the", "and", "is" o "it". Luego de aplicar la eliminación de stopwords, comenzaron a destacarse palabras relacionadas con la experiencia del usuario y las características de los productos.

Además, la personalización de stopwords permitió eliminar términos excesivamente frecuentes dentro del dominio de Amazon, como "product", "amazon" y "buy", mejorando aún más la representación del contenido relevante. Este proceso resulta beneficioso para tareas posteriores de minería de texto y clasificación, ya que reduce ruido y facilita que los modelos identifiquen patrones verdaderamente informativos.